# 投票网络可视化（Vote Network）

从 `logs/session_*/game_complete.json`（缺失时回退 `game_partial.json`）提取每一轮投票，构建“谁投给谁”的有向加权网络图：

- 节点：玩家
- 有向边 `A -> B`：表示 A 投给 B
- 边权重：累计投票次数（越粗代表投票越多）

你可以按需切换：
- 仅看最终投票快照（每轮 `votes[-1]`）
- 看所有中间投票快照（每轮 `votes` 全部）

In [ ]:
# 如果缺少 networkx，先运行本格
# %pip install networkx

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx


def project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "logs").is_dir():
        return cwd
    if (cwd.parent / "logs").is_dir():
        return cwd.parent
    return cwd


LOGS_DIR = project_root() / "logs"
USE_ONLY_FINAL_VOTE_PER_ROUND = True  # True: 每轮只用 votes[-1]；False: 用该轮所有投票快照


def iter_session_state_files(root: Path):
    """每个 session 目录只取一个状态文件：优先 complete。"""
    for d in sorted(root.glob("session_*")):
        if not d.is_dir():
            continue
        complete = d / "game_complete.json"
        partial = d / "game_partial.json"
        if complete.is_file():
            yield complete
        elif partial.is_file():
            yield partial


def collect_vote_edges(logs_dir: Path, final_only: bool = True):
    """返回投票边计数和玩家集合。

    edge_counter[(src, dst)] = src 投给 dst 的累计次数
    """
    edge_counter: Counter[tuple[str, str]] = Counter()
    players: set[str] = set()
    files_used = 0

    for path in iter_session_state_files(logs_dir):
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue

        rounds = data.get("rounds") or []
        files_used += 1

        for r in rounds:
            votes_snapshots = r.get("votes") or []
            if not votes_snapshots:
                continue

            snapshots = [votes_snapshots[-1]] if final_only else votes_snapshots
            for snap in snapshots:
                if not isinstance(snap, dict):
                    continue
                for src, dst in snap.items():
                    if src is None or dst is None:
                        continue
                    src = str(src)
                    dst = str(dst)
                    if src == dst:
                        continue
                    edge_counter[(src, dst)] += 1
                    players.add(src)
                    players.add(dst)

    return edge_counter, players, files_used

In [ ]:
edge_counter, players, n_sessions = collect_vote_edges(
    LOGS_DIR, final_only=USE_ONLY_FINAL_VOTE_PER_ROUND
)

print(f"logs 目录: {LOGS_DIR.resolve()}")
print(f"读取 session 数: {n_sessions}")
print(f"玩家数: {len(players)}")
print(f"有向边种类数: {len(edge_counter)}")
print(f"总投票条数(按累计边权求和): {sum(edge_counter.values())}")

# 看前 15 条最频繁边
top_edges = edge_counter.most_common(15)
for (src, dst), w in top_edges:
    print(f"{src} -> {dst}: {w}")

In [ ]:
# 构建有向加权图
G = nx.DiGraph()
for p in players:
    G.add_node(p)
for (src, dst), w in edge_counter.items():
    G.add_edge(src, dst, weight=w)

if G.number_of_nodes() == 0:
    print("没有可视化数据：请确认 logs 下有 rounds[].votes。")
else:
    # 固定布局保证复现性
    pos = nx.spring_layout(G, seed=42, k=1.2)

    weights = [G[u][v]["weight"] for u, v in G.edges()]
    max_w = max(weights) if weights else 1
    widths = [1.0 + 6.0 * (w / max_w) for w in weights]

    plt.figure(figsize=(10, 8))

    nx.draw_networkx_nodes(
        G,
        pos,
        node_size=1200,
        node_color="#8ecae6",
        edgecolors="#1d3557",
        linewidths=1.0,
    )
    nx.draw_networkx_labels(G, pos, font_size=10)

    nx.draw_networkx_edges(
        G,
        pos,
        width=widths,
        edge_color="#6d597a",
        arrows=True,
        arrowsize=18,
        alpha=0.75,
        connectionstyle="arc3,rad=0.12",
    )

    edge_labels = {(u, v): G[u][v]["weight"] for u, v in G.edges()}
    nx.draw_networkx_edge_labels(
        G,
        pos,
        edge_labels=edge_labels,
        font_size=8,
        label_pos=0.55,
    )

    mode = "每轮最终投票" if USE_ONLY_FINAL_VOTE_PER_ROUND else "每轮全部投票快照"
    plt.title(f"Werewolf 投票网络图（{mode}）")
    plt.axis("off")
    plt.tight_layout()
    plt.show()